# ImportVerse | تفريغ الحلقات الستة من Google Drive مجانًا

**مش محتاج Gemini Pro، ولا ترفع الفيديوهات تاني.** النوتبوك ده بيجيب الحلقات من حسابك على Google Drive بالـFile ID مباشرة، ويستخدم نموذج Whisper مفتوح المصدر لتفريغ الصوت العربي، ويحفظ النصوص في مجلد **My Drive / ImportVerse_Transcripts**.

### طريقة التشغيل
1. من Colab اختار **Runtime → Change runtime type → T4 GPU** لو متاح مجانًا؛ أو سيبه CPU (أبطأ).
2. من القائمة **Runtime → Run all**، ووافق على وصول Google Colab لحساب Drive.
3. تابع النتائج. **كل حلقة بتتحفظ فور اكتمالها**، ولو الجلسة اتقطعت افتح النوتبوك وشغّلها تاني؛ مش هيعيد الحلقات المحفوظة.
4. لما ينتهي، ارجع إلى ChatGPT وقول: **اقرأ فولدر ImportVerse_Transcripts على درايف، وراجع النصوص وابنِ الجلسة**.

**ملاحظات:** Google Colab Free مواردها محدودة ويمكن أن تنتهي الجلسة؛ نموذج التفريغ ممكن يغلط في المصطلحات الفنية، فلازم تدقيق الأسئلة قبل اعتمادها. الملفات تفضل في حساب Drive بتاعك، لكن Colab يحتاج تصريح منك لقراءتها. هينزل نسخ الفيديوهات مؤقتًا على بيئة Colab لتنفيذ التفريغ، مش هتحتاج ترفعها بنفسك.

In [ ]:
# تثبيت الأداة المفتوحة المصدر
%pip -q install -U faster-whisper google-api-python-client google-auth-httplib2

In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive', force_remount=False)

import io, json, gc
from pathlib import Path
from datetime import datetime
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import google.auth
import torch
from faster_whisper import WhisperModel

credentials, project = google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
drive_api = build('drive', 'v3', credentials=credentials, cache_discovery=False)

output = Path('/content/drive/MyDrive/ImportVerse_Transcripts')
output.mkdir(parents=True, exist_ok=True)
temporary = Path('/content/importverse_work')
temporary.mkdir(exist_ok=True)
print('📂 التفريغات هتتحفظ في:', output)
print('⚡ GPU متاح:', torch.cuda.is_available())

In [ ]:
# سيبدأ بالحلقتين 4 و5، ثم يكمل بقية الحلقات المطلوبة تلقائيًا
EPISODES = {
  "1": {
    "id": "1XUnkIyf67I0EHB7EQY8I59fg_tGUsR2T",
    "name": "Episode 1.mp4"
  },
  "2": {
    "id": "1XujgpYo9sCxZYx1ZE25dJhBikzVxwjSE",
    "name": "Episode 2.mp4"
  },
  "3": {
    "id": "1xClcDAd6B55X8u2GER3pZtuxCQqZcOnD",
    "name": "Episode 3.mp4"
  },
  "4": {
    "id": "1CRctnA7w7jL6kYbMFkzGW5DKUs1GVv5y",
    "name": "Episode 4.mp4"
  },
  "5": {
    "id": "1B6KhaddPPhYrMc75EgcGYXUnyM6SMEKp",
    "name": "Episode 5.mp4"
  },
  "6": {
    "id": "1BPdh3qNOW6nrEHeLxEAaur_Ep1uwQ8t8",
    "name": "Episode 6 final.mp4"
  }
}
ORDER = ['4','5','1','2','3','6']
FORCE_RERUN = False  # غيّرها True فقط لو عايز تعيد تفريغ حلقة مكتملة

def download_from_my_drive(file_id, dest):
    request = drive_api.files().get_media(fileId=file_id, supportsAllDrives=True)
    with open(dest, 'wb') as stream:
        downloader = MediaIoBaseDownload(stream, request, chunksize=16 * 1024 * 1024)
        done = False
        while not done:
            status, done = downloader.next_chunk(num_retries=4)
            if status:
                print(f"   تنزيل: {round(status.progress()*100)}%", end='\r')
    print(f"   ✅ تم تنزيل الفيديو مؤقتًا: {dest.stat().st_size/1024**2:.0f} MB")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
size = 'medium' if device == 'cuda' else 'small'
compute = 'float16' if device == 'cuda' else 'int8'
print('🧠 بيتم تحميل نموذج التفريغ:',size,'على',device)
model = WhisperModel(size, device=device, compute_type=compute, cpu_threads=4)

errors = {}
for num in ORDER:
    info = EPISODES[num]
    text_path = output / f'ImportVerse_Episode_{num}_Transcript.txt'
    json_path = output / f'ImportVerse_Episode_{num}_Segments.json'
    temporary_text = output / f'ImportVerse_Episode_{num}_Transcript.txt.partial'
    if text_path.exists() and text_path.stat().st_size > 200 and not FORCE_RERUN:
        print(f"⏭️ الحلقة {num} محفوظة من قبل؛ تخطي")
        continue
    local = temporary / f'Episode_{num}.mp4'
    print(f"\n🎬 الحلقة {num}: {info['name']}")
    try:
        if not local.exists() or local.stat().st_size < 1024:
            download_from_my_drive(info['id'], local)
        segments, meta = model.transcribe(
            str(local), language='ar', beam_size=3, vad_filter=True,
            condition_on_previous_text=False,
            initial_prompt='ImportVerse، نافذة، CargoX، ACI، ACID، المستورد، المصدر، بوليصة الشحن، التفويض الإلكتروني، البلوكتشين، الجمارك'
        )
        chunks = []
        with open(temporary_text,'w',encoding='utf-8') as f:
            f.write(f"ImportVerse - Episode {num}\n")
            f.write(f"Source: Google Drive file ID {info['id']}\n")
            f.write(f"Method: automated faster-whisper {size}, language=ar\n")
            f.write("Review required: names, numbers, technical details\n\n")
            for s in segments:
                row={'start':round(s.start,2),'end':round(s.end,2),'text':s.text.strip()}
                chunks.append(row)
                timestamp=f"[{int(s.start//60):02d}:{int(s.start%60):02d}]"
                f.write(timestamp+' '+row['text']+'\n')
        json_path.write_text(json.dumps(chunks,ensure_ascii=False,indent=2),encoding='utf-8')
        temporary_text.replace(text_path)  # Save only after a complete episode
        print(f"✅ تم حفظ الحلقة {num} — {len(chunks)} جزء")
    except Exception as exc:
        errors[num] = str(exc)
        print(f"❌ مشكلة الحلقة {num}: {exc}")
        temporary_text.unlink(missing_ok=True)
    finally:
        local.unlink(missing_ok=True)
        gc.collect()
print('\n✅ انتهى الدور الحالي. الملفات على Drive في:',output)
if errors:
    print('⚠️ أخطاء تحتاج مراجعة:',errors)

In [ ]:
# تحقق من الملفات بعد التشغيل
from pathlib import Path
folder=Path('/content/drive/MyDrive/ImportVerse_Transcripts')
for f in sorted(folder.glob('ImportVerse_Episode_*_Transcript.txt')):
    print(f.name, f'{f.stat().st_size/1024:.1f} KB')
print('افتح Google Drive وابحث عن: ImportVerse_Transcripts')

### بعد الانتهاء
ارجع إلى محادثتنا في ChatGPT واكتب **"شاتي، راجع التفريغات من فولدر ImportVerse_Transcripts وحدث السيشن"**. ساعتها أقرأ نصوص الحلقات من Google Drive، أراجع المعلومات وأربط أسئلة الأنشطة بالمصدر والتوقيت، وأصحّح الجلسة التدريبية.

النص المُنشأ آليًا مش معناه أن كل المعلومة مؤكدة؛ هنراجع أسماء الخدمات والأرقام والخطوات الحساسة قبل اعتماد السؤال.